# Рекурентне неуронске мреже

У претходном модулу смо обрадили богате семантичке репрезентације текста. Архитектура коју смо користили хвата агрегирано значење речи у реченици, али не узима у обзир **редослед** речи, јер операција агрегирања која следи након уграђивања уклања те информације из оригиналног текста. Пошто ови модели нису у стању да репрезентују редослед речи, они не могу решавати сложеније или амбигуне задатке као што су генерисање текста или одговарање на питања.

Да бисмо ухватили значење низа текста, користићемо архитектуру неуронске мреже која се зове **рекурентна неуронска мрежа**, или RNN. Када користимо RNN, пролазимо нашу реченицу кроз мрежу по један токен понаособ, и мрежа производи неко **стање**, које затим поново прослеђујемо мрежи заједно са следећим токеном.

![Слика која приказује пример генерисања рекурентне неуронске мреже.](../../../../../translated_images/sr/rnn.27f5c29c53d727b5.webp)

Дата улазна секвенца токена $X_0,\dots,X_n$, RNN креира низ блокова неуронске мреже и тренира овај низ „од краја до краја“ помоћу бацкпропагације. Свaki блок мреже узима пар $(X_i,S_i)$ као улаз, и као резултат производи $S_{i+1}$. Коначни статус $S_n$ или излаз $Y_n$ иде у линеарни класификатор да би произвео резултат. Сви блокови мреже деле исте тежине и тренирају се заједно једним пролазом бацкпропагације.

> Илустрација горе приказује рекурентну неуронску мрежу у „размотаном“ облику (лево), и у компактнијој рекурентној репрезентацији (десно). Важно је разумети да све RNN Ћелије имају исте **деливе тежине**.

Пошто се вектори стања $S_0,\dots,S_n$ прослеђују кроз мрежу, RNN је у стању да научи секвенцијалне зависности између речи. На пример, када се реч *not* појави негде у низу, RNN може научити да негира одређене елементе у вектору стања.

Унутар сваке RNN ћелије налазе се две матрице тежина: $W_H$ и $W_I$, и бајас $b$. На сваком RNN кораку, дати улаз $X_i$ и улазно стање $S_i$, излазно стање се израчунава као $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, где је $f$ активирајућа функција (често $\tanh$).

> За проблеме као што су генерисање текста (које ћемо обрадити у следећој јединици) или машински превод такође желимо да добијемо неку вредност излаза у сваком RNN кораку. У том случају, постоји још једна матрица $W_O$, а излаз се израчунава као $Y_i=f(W_O\times S_i+b_O)$.

Погледајмо како нам рекурентне неуронске мреже могу помоћи да класификујемо наш скуп новости.

> За „sandbox“ окружење, потребно је да покренемо следећу ћелију да бисмо били сигурни да је потребна библиотека инсталирана и да су подаци предучитани. Ако радите локално, можете прескочити следећу ћелију.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

При тренирању великих модела, алокација меморије на ГПУ-у може постати проблем. Можда ће такође бити потребно да експериментишемо са различитим величинама минипакета, тако да подаци могу да стану у нашу ГПУ меморију, али да тренинг буде довољно брз. Ако покрећете овај код на својој ГПУ машини, можете експериментисати са подешавањем величине минипакета ради убрзања тренирања.

> **Напомена**: Познато је да неке верзије NVidia драјвера не ослобађају меморију након тренирања модела. Покрећемо неколико примера у овим бележницама, и то може изазвати исцрпљивање меморије у неким подешавањима, посебно ако радите своје експерименте у оквиру исте бележнице. Ако наиђете на неке чудне грешке приликом почетка тренирања модела, можда ћете хтети да поново покренете кернел бележнице.


In [3]:
batch_size = 16
embed_size = 64

## Једноставни РНН класификатор

У случају једноставне РНН, свака рекурзивна јединица је једноставна линеарна мрежа која прима вектор улаза и вектор стања, и производи нови вектор стања. У Keras-у, ово се може представити слојем `SimpleRNN`.

Иако можемо директно проследити one-hot кодиране токене у РНН слој, то није добра идеја због велике димензионалности. Стога, користићемо embedding слој за смањење димензионалности вектора речи, затим РНН слој, и на крају `Dense` класификатор.

> **Напомена**: У случајевима када димензионалност није тако велика, на пример при коришћењу карактерске токенизације, може имати смисла директно прослеђивати one-hot кодиране токене у РНН ћелију.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Напомена:** Овде користимо неучињени уграђени слој због једноставности, али за боље резултате можемо користити унапред обучени уграђени слој користећи Word2Vec, као што је описано у претходној јединици. Било би добро вежбање за вас да прилагодите овај код да ради са унапред обученим уграђеним слојевима.

Сада хајде да обучимо наш РНН. РНН уопштено је прилично тешко обучити, јер кад се РНН ћелије развију дуж дужине секвенце, број слојева укључених у повратну пропагацију постаје прилично велики. Стога морамо изабрати мању стопу учења и обучавати мрежу на већем скупу података да бисмо постигли добре резултате. Ово може трајати прилично дуго, па је препоручљиво користити GPU.

Да бисмо убрзали ствари, обучаваћемо РНН модел само на насловима вести, изостављајући опис. Можете пробати обуку са описом и видети да ли можете да натерате модел да се обучи.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Напомена** да је прецизност овде вероватно слабија, јер тренирамо само на насловима вести.


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Сад када користимо маскирање, можемо обучавати модел на целом скупу података наслова и описа.

> **Напомена**: Да ли сте приметили да смо користили векторизатор обучен на насловима вести, а не на целом тексту чланка? Могуће је да ће неки токени бити игнорисани, па је боље поново обучити векторизатор. Међутим, то може имати веома мали утицај, па ћемо због једноставности остати при претходном обученом векторизатору.


## LSTM: Дугачка краткорочна меморија

Један од главних проблема RNN-ова је **нестајање градијената**. RNN могу бити прилично дуги и могу имати потешкоћа да пропагирају градијенте све до првог слоја мреже током бекпропагације. Када се ово деси, мрежа не може научити везе између удаљених токена. Један начин да се овај проблем избегне је увођење **експлицитног управљања стањем** коришћењем **капија**. Две најчешће архитектуре које уводе капије су **дугачка краткорочна меморија** (LSTM) и **јединица са капијама за пренос** (GRU). Овде ћемо обрадити LSTM.

![Слика која приказује пример ћелије дугачке краткорочне меморије](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

LSTM мрежа је организована слично као RNN, али постоје два стања која се преносе од слоја до слоја: стварно стање $c$, и скривени вектор $h$. У сваком јединичном елементу, скривени вектор $h_{t-1}$ се комбинује са улазом $x_t$, и заједно контролишу шта се дешава са стањем $c_t$ и излазом $h_{t}$ кроз **капије**. Свaka капија има сигмоидну активацију (излаз у опсегу $[0,1]$), коју можемо посматрати као бит-мању када се помножи са вектором стања. LSTM има следеће капије (са леве на десну у горњој слици):
* **капија за брисање** која одређује које компоненте вектора $c_{t-1}$ треба заборавити, а које пропустити.
* **капија за улаз** која одређује колико информација из вектора улаза и претходног скривеног вектора треба укључити у вектор стања.
* **капија за излаз** која узима нови вектор стања и одлучује које ће његове компоненте бити коришћене за продукцију новог скривеног вектора $h_t$.

Компоненте стања $c$ можемо сматрати заставицама које могу бити упаљене или угашене. На пример, када наиђемо на име *Алиса* у низу, претпостављамо да се односи на женско лице и подижемо заставицу у стању која каже да имамо женски именицу у реченици. Кад потом наиђемо на речи *и Том*, подижемо заставицу која каже да имамо множину именице. Тако манипулацијом са стањем можемо пратити граматичка својства реченице.

> **Напомена**: Ево одличног ресурса за разумевање унутрашњости LSTM-ова: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) аутора Кристофера Олаха.

Док унутрашња структура LSTM ћелије може изгледати компликовано, Keras сакрива ову имплементацију у слоју `LSTM`, тако да нам је у горе наведеном примеру потребно само да заменимо рекурентни слој:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Напомена** да је обука LSTM модела такође прилично спор процес, и можда нећете приметити значајан пораст тачности у почетку тренинга. Можда ћете морати наставити са тренирањем неко време да бисте постигли добру прецизност.


## Двосмерне и више-слојне РНН

У нашим примерима до сада, рекурзивне мреже раде од почетка секвенце до краја. Ово нам делује природно јер прати исти смер у којем читамо или слушамо говор. Међутим, за сценарије који захтевају насумичан приступ улазној секвенци, смисленије је извршити рекурзивне израчуне у оба смера. РНН које омогућавају израчуне у оба смера називају се **двосмерне** РНН, и може се направити омотавањем рекурзивног слоја посебним `Bidirectonal` слојем.

> **Напомена**: `Bidirectional` слој прави две копије слоја унутар себе, и поставља `go_backwards` особину једне од тих копија на `True`, што га тера да иде у супротном смеру дуж секвенце.

Рекурзивне мреже, једносмерне или двосмерне, бележе обрасце унутар секвенце и чувају их у векторе стања или их враћају као излаз. Као и код конволуцијских мрежа, можемо изградити још један рекурзивни слој након првог да бисмо уловили обрасце вишег нивоа, изграђене од обрасца нижег нивоа које је издвојио први слој. Ово нас води до појма **више-слојне РНН**, која се састоји од два или више рекурзивних мрежа, где се излаз претходног слоја прослеђује као улаз следећем слоју.

![Слика која показује вишеслојну лонг-шорт-терм-мемори РНН](../../../../../translated_images/sr/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Слика из [овог дивног поста](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) Фернанда Лопеза.*

Keras чини конструкцију ових мрежа једноставном, јер треба само додати више рекурзивних слојева моделу. За све слојеве осим последњег, потребно је да наведемо параметар `return_sequences=True`, јер треба да слој враћа сва међустања, а не само коначни резултат рекурзивног израчуна.

Хајде да направимо двосмерни LSTM са два слоја за наш проблем класификације.

> **Напомена** овај код опет траје прилично дуго да се заврши, али нам даје највишу тачност коју смо видели до сада. Дакле, можда вреди сачекати и видети резултат.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## РНН-ови за друге задатке

До сад смо се фокусирали на коришћење РНН-ова за класификацију низова текста. Међутим, они могу да обављају и много више задатака, као што су генерисање текста и машински превод &mdash; те задатке ћемо размотрити у наредној јединици.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
